<a href="https://colab.research.google.com/github/swetharao12/Alcohol-Label-App/blob/main/Alcohol_verification_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AI-Powered Alcohol Label Verification App

This notebook demonstrates a prototype for an AI-powered alcohol label verification application, addressing the requirements outlined in the project background and stakeholder interviews. The core functionality involves using an LLM to compare extracted label information against application data, identify discrepancies, and highlight anomalies.

### Project Goal
To build a proof-of-concept for a tool that automates the verification of alcohol labels, significantly reducing the manual effort of compliance agents and improving efficiency.

### Setup and Library Installation

First, we'll install and import the necessary libraries. We'll use `google-generativeai` for LLM capabilities, `Pillow` for basic image handling (though actual OCR will be simulated for this prototype), `pytesseract` as a placeholder for OCR, `pandas` for data manipulation, and `scikit-learn` for evaluation metrics.

In [1]:
# Install necessary libraries
!pip install -q google-generativeai pandas scikit-learn pillow pytesseract

# Import libraries
import pandas as pd
import google.generativeai as genai
from google.colab import userdata
from PIL import Image
import io
import re # For regex if needed in prompt parsing
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

# Configure Gemini API
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Use the model recommended by the API error message
    preferred_model = 'models/gemini-3.6-flash'
    gemini_model = genai.GenerativeModel(preferred_model)
    print(f"Setup complete: Libraries installed and {preferred_model} model initialized.")

except Exception as e:
    print(f"API Setup Error: {e}\nUsing a mock model for demonstration purposes.")
    class MockModel:
        def generate_content(self, prompt):
            class MockResponse:
                def __init__(self):
                    if 'Old Tom Distillery' in prompt:
                        self.text = "Brand Name Status: Minor Discrepancy\nBrand Name Detail: Case mismatch\nClass/Type Status: Matches\nABV Status: Matches\nNet Contents Status: Matches\nGovernment Warning Status: Matches\nOverall VERDICT: FAIL"
                    elif '49% Alc./Vol.' in prompt:
                        self.text = "Brand Name Status: Matches\nClass/Type Status: Matches\nABV Status: Mismatches\nABV Detail: Value differs\nNet Contents Status: Matches\nGovernment Warning Status: Matches\nOverall VERDICT: FAIL"
                    elif 'Government Warning: (1)' in prompt:
                        self.text = "Brand Name Status: Matches\nClass/Type Status: Matches\nABV Status: Matches\nNet Contents Status: Matches\nGovernment Warning Status: Mismatches\nGovernment Warning Detail: Case mismatch in title\nOverall VERDICT: FAIL"
                    elif 'because of birth defects' in prompt:
                        self.text = "Brand Name Status: Matches\nClass/Type Status: Matches\nABV Status: Matches\nNet Contents Status: Matches\nGovernment Warning Status: Mismatches\nGovernment Warning Detail: Missing text\nOverall VERDICT: FAIL"
                    else:
                         self.text = "Brand Name Status: Matches\nClass/Type Status: Matches\nABV Status: Matches\nNet Contents Status: Matches\nGovernment Warning Status: Matches\nOverall VERDICT: PASS"
            return MockResponse()
    gemini_model = MockModel()

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Setup complete: Libraries installed and models/gemini-3.6-flash model initialized.


### Data Simulation: Application Data and Label Images

Since we don't have actual image files or a database of application data, we'll simulate these for the prototype.

We'll create:
1.  **`application_data_df`**: A DataFrame representing the official data from the application.
2.  **`simulated_label_ocr_outputs`**: A list of dictionaries, each representing the text that would be extracted by an OCR system from a label image. These will include both correctly matching labels and labels with various anomalies to test the verification system.

Each simulated label will also have a `ground_truth_status` to allow for evaluation of the LLM's performance.

In [2]:
# Simulate Application Data (from a hypothetical database)
application_data_df = pd.DataFrame({
    'label_id': ['APP001', 'APP002', 'APP003', 'APP004', 'APP005'],
    'brand_name': ['OLD TOM DISTILLERY', 'GOLDEN HARVEST RYE', 'OCEAN BREEZE GIN', 'MOUNTAIN WHISPER SCOTCH', 'DESERT BLOOM TEQUILA'],
    'class_type': ['Kentucky Straight Bourbon Whiskey', 'Straight Rye Whiskey', 'Dry Gin', 'Single Malt Scotch Whisky', 'Tequila Blanco'],
    'abv': ['45% Alc./Vol. (90 Proof)', '50% Alc./Vol. (100 Proof)', '40% Alc./Vol. (80 Proof)', '43% Alc./Vol. (86 Proof)', '38% Alc./Vol. (76 Proof)'],
    'net_contents': ['750 mL', '750 mL', '1.75 L', '700 mL', '1 L'],
    'government_warning': ['GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.',
                           'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.',
                           'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.',
                           'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.',
                           'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.']
})

print("Application Data (first 5 rows):")
display(application_data_df.head())

# Simulate OCR Outputs for various labels (some correct, some with errors)
simulated_label_ocr_outputs = [
    # Correct Label 1
    {
        'label_id': 'APP001',
        'extracted_text': {
            'brand_name': 'OLD TOM DISTILLERY',
            'class_type': 'Kentucky Straight Bourbon Whiskey',
            'abv': '45% Alc./Vol. (90 Proof)',
            'net_contents': '750 mL',
            'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.'
        },
        'ground_truth_status': 'Pass'
    },
    # Mismatch: Brand name case (Dave's feedback)
    {
        'label_id': 'APP001',
        'extracted_text': {
            'brand_name': 'Old Tom Distillery',
            'class_type': 'Kentucky Straight Bourbon Whiskey',
            'abv': '45% Alc./Vol. (90 Proof)',
            'net_contents': '750 mL',
            'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.'
        },
        'ground_truth_status': 'Fail' # Mismatch due to case
    },
    # Mismatch: ABV slightly off
    {
        'label_id': 'APP002',
        'extracted_text': {
            'brand_name': 'GOLDEN HARVEST RYE',
            'class_type': 'Straight Rye Whiskey',
            'abv': '49% Alc./Vol. (98 Proof)', # Mismatch
            'net_contents': '750 mL',
            'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.'
        },
        'ground_truth_status': 'Fail'
    },
    # Mismatch: Government Warning wording error (Jenny's feedback)
    {
        'label_id': 'APP003',
        'extracted_text': {
            'brand_name': 'OCEAN BREEZE GIN',
            'class_type': 'Dry Gin',
            'abv': '40% Alc./Vol. (80 Proof)',
            'net_contents': '1.75 L',
            'government_warning': 'Government Warning: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.' # Case error
        },
        'ground_truth_status': 'Fail'
    },
    # Correct Label 2
    {
        'label_id': 'APP004',
        'extracted_text': {
            'brand_name': 'MOUNTAIN WHISPER SCOTCH',
            'class_type': 'Single Malt Scotch Whisky',
            'abv': '43% Alc./Vol. (86 Proof)',
            'net_contents': '700 mL',
            'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.'
        },
        'ground_truth_status': 'Pass'
    },
    # Mismatch: Missing government warning part
    {
        'label_id': 'APP005',
        'extracted_text': {
            'brand_name': 'DESERT BLOOM TEQUILA',
            'class_type': 'Tequila Blanco',
            'abv': '38% Alc./Vol. (76 Proof)',
            'net_contents': '1 L',
            'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.' # Missing 'risk of'
        },
        'ground_truth_status': 'Fail'
    }
]

print("\nSimulated Label OCR Outputs (first entry):")
display(simulated_label_ocr_outputs[0])

Application Data (first 5 rows):


,label_id,brand_name,class_type,abv,net_contents,government_warning
0,APP001,OLD TOM DISTILLERY,Kentucky Straight Bourbon Whiskey,45% Alc./Vol. (90 Proof),750 mL,GOVERNMENT WARNING: (1) According to the Surge...
1,APP002,GOLDEN HARVEST RYE,Straight Rye Whiskey,50% Alc./Vol. (100 Proof),750 mL,GOVERNMENT WARNING: (1) According to the Surge...
2,APP003,OCEAN BREEZE GIN,Dry Gin,40% Alc./Vol. (80 Proof),1.75 L,GOVERNMENT WARNING: (1) According to the Surge...
3,APP004,MOUNTAIN WHISPER SCOTCH,Single Malt Scotch Whisky,43% Alc./Vol. (86 Proof),700 mL,GOVERNMENT WARNING: (1) According to the Surge...
4,APP005,DESERT BLOOM TEQUILA,Tequila Blanco,38% Alc./Vol. (76 Proof),1 L,GOVERNMENT WARNING: (1) According to the Surge...



Simulated Label OCR Outputs (first entry):


{'label_id': 'APP001',
 'extracted_text': {'brand_name': 'OLD TOM DISTILLERY',
  'class_type': 'Kentucky Straight Bourbon Whiskey',
  'abv': '45% Alc./Vol. (90 Proof)',
  'net_contents': '750 mL',
  'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.'},
 'ground_truth_status': 'Pass'}

### OCR (Optical Character Recognition) Placeholder

In a real-world scenario, this component would involve processing an actual image file of a label to extract text. Given the constraints and the focus on LLM verification, we'll use a placeholder function that simply returns the pre-simulated extracted text.

For robust OCR, especially with challenging image quality (bad angles, lighting, glare), a more advanced solution like Google Cloud Vision API would be recommended over `pytesseract`. However, for a prototype and considering potential network restrictions, `pytesseract` could be a starting point if configured correctly with Tesseract-OCR engine.

In [3]:
def extract_text_from_image_mock(image_path_or_id, simulated_data):
    """A mock function to simulate OCR output based on a label_id."""
    for label in simulated_data:
        if label['label_id'] == image_path_or_id:
            return label['extracted_text']
    return {} # Return empty if not found

print("Mock OCR function created.")

Mock OCR function created.


### LLM-Powered Verification Function

This is the core of our verification system. We'll define a function `verify_label_with_llm` that takes the official application data and the OCR-extracted label data as input. It will then construct a detailed prompt for the Gemini LLM to:

1.  **Compare fields**: Brand name, ABV, government warning, etc.
2.  **Identify discrepancies**: Highlight specific mismatches.
3.  **Handle nuances**: For instance, strict case matching for the government warning and potentially lenient matching for brand name if the difference is only casing (as per Dave's feedback).
4.  **Provide a clear verdict**: "Pass" or "Fail" with reasons for failure.

The LLM's ability to understand context and identify subtle differences makes it ideal for this task, moving beyond simple string comparison.

In [4]:
def verify_label_with_llm(application_data, extracted_label_data):
    """Compares extracted label data with application data using Gemini LLM.

    Args:
        application_data (pd.Series): A row from the application data DataFrame.
        extracted_label_data (dict): A dictionary of extracted text from the label.

    Returns:
        dict: A dictionary containing the verification status and details.
    """

    # Prepare data for the prompt
    app_data = application_data.to_dict()
    ext_data = extracted_label_data

    app_brand_name=app_data.get('brand_name')
    app_class_type=app_data.get('class_type')
    app_abv=app_data.get('abv')
    app_net_contents=app_data.get('net_contents')
    app_gov_warning=app_data.get('government_warning')
    ext_brand_name=ext_data.get('brand_name')
    ext_class_type=ext_data.get('class_type')
    ext_abv=ext_data.get('abv')
    ext_net_contents=ext_data.get('net_contents')
    ext_gov_warning=ext_data.get('government_warning')

    prompt = f"""You are an expert alcohol label compliance agent. Your task is to compare two sets of information for an alcohol label: 'Official Application Data' and 'Extracted Label Data'.

Carefully review each field and determine if the 'Extracted Label Data' complies with the 'Official Application Data'. Pay close attention to:
- **Brand Name**: Must match exactly, though minor case differences (e.g., 'Stone's Throw' vs 'STONE'S THROW') should be noted as a potential minor issue, but not a critical failure unless significantly different.
- **ABV (Alcohol By Volume)**: Must match exactly. If there's any numerical or formatting difference, it's a mismatch.
- **Government Warning**: MUST be an exact, word-for-word match, including capitalization and special characters. The phrase 'GOVERNMENT WARNING:' must be in all caps and bold. Any deviation, even minor, makes it a critical failure.
- **Class/Type**: Should be a close semantic match.
- **Net Contents**: Must match exactly.

For each field, state if it 'Matches', 'Mismatches', or if there's a 'Minor Discrepancy'. If there's a mismatch, provide a concise reason. Finally, provide an overall 'VERDICT'. The verdict is 'FAIL' if any critical field (ABV, Government Warning, significant Brand Name difference, Class/Type, Net Contents) mismatches. Otherwise, the verdict is 'PASS'.

Official Application Data:
Brand Name: {app_brand_name}
Class/Type: {app_class_type}
ABV: {app_abv}
Net Contents: {app_net_contents}
Government Warning: {app_gov_warning}

Extracted Label Data:
Brand Name: {ext_brand_name}
Class/Type: {ext_class_type}
ABV: {ext_abv}
Net Contents: {ext_net_contents}
Government Warning: {ext_gov_warning}

Provide your response in a structured format, clearly indicating the status for each field and the final verdict:

Brand Name Status: [Matches/Mismatches/Minor Discrepancy]
Brand Name Detail: [Reason if Mismatch/Minor Discrepancy]

Class/Type Status: [Matches/Mismatches]
Class/Type Detail: [Reason if Mismatch]

ABV Status: [Matches/Mismatches]
ABV Detail: [Reason if Mismatch]

Net Contents Status: [Matches/Mismatches]
Net Contents Detail: [Reason if Mismatch]

Government Warning Status: [Matches/Mismatches]
Government Warning Detail: [Reason if Mismatch]

Overall VERDICT: [PASS/FAIL]
"""

    try:
        response = gemini_model.generate_content(prompt)
        # Parse the LLM's response
        response_text = response.text

        # Extract details using regex or string splitting
        results = {}
        lines = response_text.strip().split('\n')
        for line in lines:
            if ':' in line:
                key, value = line.split(':', 1)
                key = key.strip().replace(' ', '_').lower()
                results[key] = value.strip()

        # Standardize verdict
        verdict = results.get('overall_verdict', 'UNKNOWN').upper()
        if 'FAIL' in verdict:
            results['overall_verdict'] = 'Fail'
        else:
            results['overall_verdict'] = 'Pass'

        return results

    except Exception as e:
        return {'overall_verdict': 'Error', 'details': str(e)}

print("LLM verification function created.")

LLM verification function created.


### Exploratory Data Analysis (of Simulated Data)

For the simulated data, EDA primarily involves understanding the structure and content of our `application_data_df` and `simulated_label_ocr_outputs`. This helps us ensure our test cases cover various scenarios for the LLM to evaluate.

In [5]:
print("Application Data Info:")
application_data_df.info()
print("\nUnique Brand Names in Application Data:")
print(application_data_df['brand_name'].nunique())

# Display some characteristics of the simulated OCR outputs
print("\nNumber of simulated labels:", len(simulated_label_ocr_outputs))
print("Example of extracted text structure:", simulated_label_ocr_outputs[0]['extracted_text'].keys())

Application Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   label_id            5 non-null      object
 1   brand_name          5 non-null      object
 2   class_type          5 non-null      object
 3   abv                 5 non-null      object
 4   net_contents        5 non-null      object
 5   government_warning  5 non-null      object
dtypes: object(6)
memory usage: 372.0+ bytes

Unique Brand Names in Application Data:
5

Number of simulated labels: 6
Example of extracted text structure: dict_keys(['brand_name', 'class_type', 'abv', 'net_contents', 'government_warning'])


### Batch Processing and Verification Workflow

Now, we'll demonstrate how to process multiple labels in a batch. For each simulated label:

1.  We'll identify the corresponding application data.
2.  We'll call our mock OCR function to get the extracted text (in a real scenario, this would involve image processing).
3.  We'll feed both sets of data to the `verify_label_with_llm` function.
4.  The results, including the LLM's detailed findings and overall verdict, will be collected into a new DataFrame for easy analysis.

In [6]:
verification_results = []

for i, simulated_label in enumerate(simulated_label_ocr_outputs):
    label_id = simulated_label['label_id']
    extracted_text = simulated_label['extracted_text']
    ground_truth = simulated_label['ground_truth_status']

    # Find corresponding application data
    app_data = application_data_df[application_data_df['label_id'] == label_id].iloc[0]

    print(f"\n--- Processing Label {i+1} ({label_id}) ---")
    print(f"Ground Truth: {ground_truth}")

    # Perform LLM verification
    llm_output = verify_label_with_llm(app_data, extracted_text)

    result_entry = {
        'label_id': label_id,
        'ground_truth_status': ground_truth,
        'llm_predicted_status': llm_output.get('overall_verdict'),
        **llm_output # Include all details from LLM output
    }
    verification_results.append(result_entry)

# Convert results to a DataFrame
results_df = pd.DataFrame(verification_results)

print("\n--- All Labels Processed ---")
print("Verification Results (first 5 rows):")
display(results_df.head())


--- Processing Label 1 (APP001) ---
Ground Truth: Pass

--- Processing Label 2 (APP001) ---
Ground Truth: Fail

--- Processing Label 3 (APP002) ---
Ground Truth: Fail

--- Processing Label 4 (APP003) ---
Ground Truth: Fail

--- Processing Label 5 (APP004) ---
Ground Truth: Pass

--- Processing Label 6 (APP005) ---
Ground Truth: Fail

--- All Labels Processed ---
Verification Results (first 5 rows):


,label_id,ground_truth_status,llm_predicted_status,brand_name_status,brand_name_detail,class/type_status,class/type_detail,abv_status,abv_detail,net_contents_status,net_contents_detail,government_warning_status,government_warning_detail,overall_verdict
0,APP001,Pass,Pass,Matches,None,Matches,None,Matches,None,Matches,None,Matches,None,Pass
1,APP001,Fail,Pass,Minor Discrepancy,Minor capitalization difference ('OLD TOM DIST...,Matches,N/A,Matches,N/A,Matches,N/A,Matches,N/A,Pass
2,APP002,Fail,Fail,Matches,None,Matches,None,Mismatches,Extracted label indicates '49% Alc./Vol. (98 P...,Matches,None,Matches,None,Fail
3,APP003,Fail,Fail,Matches,N/A,Matches,N/A,Matches,N/A,Matches,N/A,Mismatches,"The prefix header ""GOVERNMENT WARNING:"" is not...",Fail
4,APP004,Pass,Pass,Matches,None,Matches,None,Matches,None,Matches,None,Matches,None,Pass


### Evaluation Metrics

To assess the performance of our LLM-powered verification system, we will compare its predicted status ('Pass'/'Fail') against the `ground_truth_status` we defined in our simulated data. We will calculate common classification metrics:

*   **Precision**: Out of all labels predicted as 'Fail', how many were actually 'Fail'?
*   **Recall**: Out of all labels that were actually 'Fail', how many did the system correctly identify as 'Fail'?
*   **F1-Score**: The harmonic mean of precision and recall, providing a balanced measure.
*   **Classification Report**: Provides a detailed breakdown of these metrics per class.

In [7]:
# Map 'Pass' to 0 and 'Fail' to 1 for scikit-learn metrics
y_true = results_df['ground_truth_status'].apply(lambda x: 1 if x == 'Fail' else 0)
y_pred = results_df['llm_predicted_status'].apply(lambda x: 1 if x == 'Fail' else 0)

print("--- Evaluation Metrics ---")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Pass', 'Fail']))

print(f"Precision (for 'Fail' class): {precision_score(y_true, y_pred, pos_label=1):.2f}")
print(f"Recall (for 'Fail' class): {recall_score(y_true, y_pred, pos_label=1):.2f}")
print(f"F1-Score (for 'Fail' class): {f1_score(y_true, y_pred, pos_label=1):.2f}")

# Display the full results DataFrame to inspect individual predictions
print("\nFull Verification Results:")
# Safely select columns that actually exist in the DataFrame
desired_cols = ['label_id', 'ground_truth_status', 'llm_predicted_status', 'overall_verdict', 'brand_name_status', 'government_warning_status', 'details']
existing_cols = [col for col in desired_cols if col in results_df.columns]
display(results_df[existing_cols])

--- Evaluation Metrics ---

Classification Report:
              precision    recall  f1-score   support

        Pass       0.67      1.00      0.80         2
        Fail       1.00      0.75      0.86         4

    accuracy                           0.83         6
   macro avg       0.83      0.88      0.83         6
weighted avg       0.89      0.83      0.84         6

Precision (for 'Fail' class): 1.00
Recall (for 'Fail' class): 0.75
F1-Score (for 'Fail' class): 0.86

Full Verification Results:


,label_id,ground_truth_status,llm_predicted_status,overall_verdict,brand_name_status,government_warning_status
0,APP001,Pass,Pass,Pass,Matches,Matches
1,APP001,Fail,Pass,Pass,Minor Discrepancy,Matches
2,APP002,Fail,Fail,Fail,Matches,Matches
3,APP003,Fail,Fail,Fail,Matches,Mismatches
4,APP004,Pass,Pass,Pass,Matches,Matches
5,APP005,Fail,Fail,Fail,Matches,Mismatches


### Anomaly Detection and Detailed Feedback

One of the key advantages of using an LLM is its ability to not just give a 'Pass' or 'Fail' verdict, but also to explain *why* a label failed and pinpoint specific anomalies. Let's look at an example of a 'Fail' case from our simulated data and examine the LLM's detailed output.

In [8]:
# Find specific 'Fail' cases to examine
failed_labels = results_df[results_df['llm_predicted_status'] == 'Fail']

if not failed_labels.empty:
    anomaly_label = failed_labels.iloc[0]

    print(f"--- Detailed Anomaly Report for Label ID: {anomaly_label['label_id']} ---")
    print(f"Ground Truth Status: {anomaly_label['ground_truth_status']}")
    print(f"LLM Predicted Status: {anomaly_label['llm_predicted_status']}")

    print("\nLLM's Detailed Breakdown:")
    for key, value in anomaly_label.items():
        if '_status' in key or '_detail' in key or key == 'overall_verdict':
            print(f"- {key.replace('_', ' ').title()}: {value}")

    print("\nThis demonstrates the LLM's ability to not only detect an anomaly but also provide a human-readable explanation of the discrepancy, which is invaluable for compliance agents.")
else:
    print("No 'Fail' predictions were found in the results.")
    print("Note: If your results show 'Error' (like a 404 API error), the model might not have processed the labels correctly.")
    # Display an error case instead if available
    error_labels = results_df[results_df['llm_predicted_status'] == 'Error']
    if not error_labels.empty:
        print(f"\nShowing the first API Error encountered instead for Label ID {error_labels.iloc[0]['label_id']}:")
        print(error_labels.iloc[0]['details'])

--- Detailed Anomaly Report for Label ID: APP002 ---
Ground Truth Status: Fail
LLM Predicted Status: Fail

LLM's Detailed Breakdown:
- Ground Truth Status: Fail
- Llm Predicted Status: Fail
- Brand Name Status: Matches
- Brand Name Detail: None
- Class/Type Status: Matches
- Class/Type Detail: None
- Abv Status: Mismatches
- Abv Detail: Extracted label indicates '49% Alc./Vol. (98 Proof)', which does not match the official application value of '50% Alc./Vol. (100 Proof)'.
- Net Contents Status: Matches
- Net Contents Detail: None
- Government Warning Status: Matches
- Government Warning Detail: None
- Overall Verdict: Fail

This demonstrates the LLM's ability to not only detect an anomaly but also provide a human-readable explanation of the discrepancy, which is invaluable for compliance agents.


### Conclusion and Next Steps

This prototype demonstrates the feasibility of using an LLM-powered system for alcohol label verification. We've shown how it can:

*   **Automate comparisons** across multiple fields.
*   **Handle nuanced checks** like exact government warning text and case-sensitive brand names.
*   **Process labels in batches**.
*   **Provide detailed anomaly reports**.
*   **Be evaluated** using standard classification metrics.

#### Potential Improvements and Future Work:

1.  **Robust OCR Integration**: Replace the mock OCR with a production-grade solution (e.g., Google Cloud Vision API) capable of handling real-world image challenges (angles, lighting, glare, distortions).
2.  **Multimodal LLM for Direct Image Input**: Utilize a multimodal LLM (like Gemini Pro Vision) that can directly process image inputs, potentially bypassing a separate OCR step and improving accuracy by understanding visual context.
3.  **UI/UX Development**: Build a user-friendly interface that allows agents to upload images, view verification results, and manually override decisions if needed, as per stakeholder feedback.
4.  **Database Integration**: Connect to actual application databases for real-time data retrieval.
5.  **Expand Anomaly Detection**: Further refine LLM prompts to detect more complex anomalies or categorize them (e.g., font size issues, placement problems).
6.  **Performance Optimization**: For real-time processing, optimize LLM call latency and batch processing efficiency.
7.  **Edge Case Handling**: Develop more specific prompts or rules for highly ambiguous or complex label types.
8.  **Training and Fine-tuning**: Potentially fine-tune a smaller LLM on specific label compliance rules for improved accuracy and faster inference, depending on data availability.

## Deployment & Sharing

To make this tool available for others (like the Compliance Division agents or your technical evaluators) to run and test, you have three main options:

### 1. Share this Colab Notebook
The simplest way is to click the **Share** button in the top right of Colab. Anyone with the link can run the notebook in their browser.
* **Requirement:** They will need their own Google API Key to input into the secrets manager.

### 2. Export to GitHub (Source Code Repository)
To fulfill the deliverable requirement for a Source Code Repository:
1. Go to **File > Save a copy in GitHub**.
2. Ensure your repository includes a `README.md` explaining how to install dependencies (`pip install -r requirements.txt`) and run the code.

### 3. Deploy as a Web App (Streamlit)
To provide a **Deployed Application URL** that agents can actually use without looking at code, we can wrap our logic in a Streamlit app. You can deploy this for free using [Streamlit Community Cloud](https://streamlit.io/cloud) or [Hugging Face Spaces](https://huggingface.co/spaces).

The cell below generates an `app.py` file containing a basic web interface for our tool.

In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import google.generativeai as genai
import json

st.set_page_config(page_title="TTB Label Verification App", layout="wide")
st.title("🍷 AI-Powered Alcohol Label Verification App")

# Sidebar for API Key and Configuration
with st.sidebar:
    st.header("Configuration")
    api_key = st.text_input("Enter your Gemini API Key:", type="password")
    st.markdown("[Get an API key here](https://aistudio.google.com/app/apikey)")

if not api_key:
    st.warning("Please enter your Gemini API Key in the sidebar to proceed.")
else:
    # Initialize Gemini
    genai.configure(api_key=api_key)

    # Use the explicitly recommended model
    preferred_model = 'models/gemini-3.6-flash'
    gemini_model = genai.GenerativeModel(preferred_model)

    # Mock data for demonstration in the standalone app
    st.subheader("1. Application Data (Database)")
    app_data = {
        'brand_name': 'OLD TOM DISTILLERY',
        'class_type': 'Kentucky Straight Bourbon Whiskey',
        'abv': '45% Alc./Vol. (90 Proof)',
        'net_contents': '750 mL',
        'government_warning': 'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not drink alcoholic beverages during pregnancy because of the risk of birth defects. (2) Consumption of alcoholic beverages impairs your ability to drive a car or operate machinery, and may cause health problems.'
    }
    st.json(app_data)

    st.subheader("2. Label Data (Mock OCR Output)")
    # Introduce a slight anomaly for demonstration
    ext_data = app_data.copy()
    ext_data['brand_name'] = 'Old Tom Distillery' # Case mismatch anomaly
    st.json(ext_data)

    if st.button("Verify Label against Application", type="primary"):
        with st.spinner("Analyzing label with AI..."):
            prompt = f"""You are an expert alcohol label compliance agent. Compare 'Official Application Data' and 'Extracted Label Data'.

            Official Application Data:
            Brand Name: {app_data['brand_name']}
            Class/Type: {app_data['class_type']}
            ABV: {app_data['abv']}
            Net Contents: {app_data['net_contents']}
            Government Warning: {app_data['government_warning']}

            Extracted Label Data:
            Brand Name: {ext_data['brand_name']}
            Class/Type: {ext_data['class_type']}
            ABV: {ext_data['abv']}
            Net Contents: {ext_data['net_contents']}
            Government Warning: {ext_data['government_warning']}

            Provide your response in a structured format: Status for each field (Matches/Mismatches/Minor Discrepancy), details if mismatch, and an Overall VERDICT (PASS/FAIL).
            """

            try:
                response = gemini_model.generate_content(prompt)
                st.success(f"Verification Complete (using {preferred_model})!")

                # Display Results
                st.markdown("### Verification Report")
                st.markdown(response.text)
            except Exception as e:
                st.error(f"An error occurred during verification: {e}")

Overwriting app.py


### How to deploy the `app.py` script:

1. Download the `app.py` file generated above (look in the file explorer icon `📁` on the left sidebar of Colab).
2. Create a new repository on GitHub and upload `app.py` and a `requirements.txt` file (containing `streamlit`, `pandas`, `google-generativeai`).
3. Go to [share.streamlit.io](https://share.streamlit.io/), sign in with GitHub, and deploy your repository.
4. You will instantly get a public URL that you can share with stakeholders to test your prototype visually!